# 01 · Masks

Turn the tap points saved on the phone into masks with **SAM 2**, then review them.

1. SAM 2.1 runs once per tapped object, with the tap as a positive point.
2. You look through the grid and mark each sample **accepted** or **rejected**.
3. A bad mask can be fixed with extra positive / negative clicks.
4. `labels.png`, `instances.png`, the extra clicks and `mask_status` are written back into each zip.

SAM is used only here, to make training labels. It is never used at inference.
The first run downloads the SAM 2.1 weights from Hugging Face (about 180 MB for *small*).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "python"))  # use the package without installing it
DATA = Path("../data")      # session zips exported from the phone
MODELS = Path("../models")  # checkpoints, calibration.json, UNetUltra.mlpackage
MODELS.mkdir(exist_ok=True)

In [ ]:
from unetultra.session import find_sessions
from unetultra.validate import validate_session

ZIPS = find_sessions(DATA)          # or list the zips you want: ["../data/session_....zip"]
for path in ZIPS:
    print(validate_session(path).format())

## Run SAM 2 from the tap points

Samples that already have masks are left alone; pass `redo=True` to start again.

In [ ]:
from unetultra.masks import Sam2Predictor

# CPU is the safe choice (a second or two per object). device="mps" can be faster but has been seen to stall.
predictor = Sam2Predictor("facebook/sam2.1-hiera-small", device="cpu")   # or "facebook/sam2.1-hiera-base-plus"
print("SAM 2 running on", predictor.device)

In [ ]:
from unetultra.masks import MaskReview

import time

review = MaskReview(ZIPS, predictor)
started = time.time()
# One line per sample, printed before SAM runs on it, so a stall is visible straight away.
review.run(progress=lambda done, total, sample_id: print(
    f"{done}/{total}  sample {sample_id}  ({time.time() - started:.0f}s so far)", flush=True))
print(f"done in {time.time() - started:.0f}s")
review.counts()

## Review

Each tile shows the mask tint and the tap (yellow cross). Set every sample to **accepted** or
**rejected**; change `page` to move through the session, or pass `only="pending"`.
Background-only samples have an empty mask and are accepted automatically.

In [ ]:
from unetultra.review_ui import review_grid

review_grid(review, page=0, per_page=12)

## Fix a mask

Pick a sample and object, choose *add to mask* or *remove from mask*, and click on the image.
SAM re-runs with every click. The clicks are saved in `meta.json` as `extra_points`.

In [ ]:
%matplotlib widget
from unetultra.review_ui import fix_mask

item = next(item for item in review.items if item.meta["objects"])   # or review[(ZIPS[0], "0007")]
fix_mask(review, item.key, object_index=0)

## Save

Nothing is written until this cell runs. Uncomment the first line to accept everything still pending.

In [ ]:
# review.accept_all_pending()
print(review.save())
for path in ZIPS:
    print(validate_session(path).format())

## Check against the fake session's known shapes

Only meaningful for zips made by `tools/make_fake_session.py`: the true masks are regenerated
from the same seed and compared with SAM's. The target is Dice > 0.95.

In [ ]:
import tempfile
from unetultra.fake_session import make_fake_session
from unetultra.masks import dice
from unetultra.session import Session

for path in ZIPS:
    with Session(path) as made:
        if made.manifest["app_version"] != "fake":
            continue
        seed = int(made.session_id.removeprefix("fake"))
        size = tuple(made.meta(made.sample_ids[0])["image_size"])
        with tempfile.TemporaryDirectory() as tmp:
            truth_zip = make_fake_session(Path(tmp) / "truth.zip", n_samples=len(made.sample_ids), seed=seed,
                                          split=made.split, classes=made.classes, image_size=size, with_masks=True)
            with Session(truth_zip) as truth:
                scores = [dice(made.instances(s) > 0, truth.instances(s) > 0)
                          for s in made.sample_ids if made.instances(s) is not None and truth.meta(s)["objects"]]
        print(f"{path}: Dice min {min(scores):.3f}, mean {sum(scores) / len(scores):.3f} over {len(scores)} samples")